In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
import statsmodels.formula.api as smf

# Load the data
df_flights = pd.read_csv('vuelos_kayak_clean_final.csv')

# --- Analysis 1: The "Distance" Test (Regression) ---

# 1. Calculate Pearson Correlation
correlation, p_value_corr = pearsonr(df_flights['Distance_km'], df_flights['precio'])

# 2. Run Linear Regression (Price = a * Distance + b)
model = smf.ols('precio ~ Distance_km', data=df_flights).fit()

# 3. Print the results summary (only the key coefficients)
print("--- Analysis 1: The 'Distance' Test (Regression) ---")
print(f"Pearson Correlation (Distance vs. Price): {correlation:.4f} (P-value: {p_value_corr:.4f})")
print("\nOLS Regression Summary:")
print(model.summary().tables[1]) # Print the coefficients table

# 4. Create a scatter plot with the regression line
plt.figure(figsize=(10, 6))
sns.regplot(x='Distance_km', y='precio', data=df_flights,
            line_kws={"color": "red", "label": "Regression Line"})
plt.title('Flight Price vs. Distance with Regression Line')
plt.xlabel('Distance (km)')
plt.ylabel('Price (precio)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.savefig('price_vs_distance_regression.png')
# Close the plot to free memory
plt.close()

print("\nSaved plot: price_vs_distance_regression.png")

--- Analysis 1: The 'Distance' Test (Regression) ---
Pearson Correlation (Distance vs. Price): 0.1739 (P-value: 0.0000)

OLS Regression Summary:
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept     108.1492      0.590    183.408      0.000     106.993     109.305
Distance_km     0.0193      0.000     50.447      0.000       0.019       0.020

Saved plot: price_vs_distance_regression.png


**Super DEAL**

In [ ]:
# Z SCORE
city_price_stats = df_flights.groupby('Dest_City')['precio'].agg(['mean', 'std']).reset_index()
city_price_stats.rename(columns={'mean': 'City_Mean_Price', 'std': 'City_Std_Price'}, inplace=True)

df_flights = pd.merge(df_flights, city_price_stats, on='Dest_City', how='left')
df_flights['Z_Score'] = (df_flights['precio'] - df_flights['City_Mean_Price']) / df_flights['City_Std_Price'].replace(0, np.nan)

# 3. Super Deals (Z-Score < -2.0)
super_deals = df_flights[df_flights['Z_Score'] < -2]

# df_flights.to_csv('european_flight_data_with_zscore.csv', index=False)

super_deals # no promising resultss at first sight

,Date,Month,Day_Name,Weekday,Departure_Time,Arrival_Time,Duration_Minutes,Distance_km,precio,Stops_Numeric,...,origen_iata,destino_iata,Origin_Country,Dest_Country,Route,Origin_City,Dest_City,City_Mean_Price,City_Std_Price,Z_Score
